#### async (test)

In [ ]:
///- --test static --symbol

:< testing

In [ ]:
:< rust_operators

:= base_let' x =
    let' x

:! future t =
    `(
        rust::rust_type
            `($'std_future_Future<`t>')
            `($'std::future::Future<Output = `t>')
            "type std_future_Future<'T> = class end"
    )

:== future_pin t = rust::pin (rust::box (rust::dyn' (future t)))

:== future_pin_send t = rust::pin (rust::box (rust::send (rust::dyn' (future t))))

:= block_on_tokio \* t. (fn : future_pin t) : t =
    := runtime : infer =
        !\($'$"tokio::runtime::Builder::new_multi_thread().enable_all().build().unwrap()"')
    !\\(fn, $'"!runtime.handle().block_on($0)"')

:= block_on_futures \* t. (fn : future_pin t) : t =
    !\\(fn, $'"futures::executor::block_on($0)"')

:! stream_ext_tokio =
    `(
        rust::rust_type
            `($'tokio_stream_StreamExt')
            `($'tokio_stream::StreamExt')
            "type tokio_stream_StreamExt = class end"
    )

:! join_handle_tokio t =
    `(
        rust::rust_type
            `($'tokio_task_JoinHandle<`t>')
            `($'tokio::task::JoinHandle<`t>')
            "type tokio_task_JoinHandle<'T> = class end"
    )

:= stream_collect_futures \* t u.
    (stream : t)
    : future_pin (am'::vec u)
    =
    !\($'"Box::pin(futures::stream::StreamExt::collect(!stream))"')

:= stream_filter_map_futures \* t u v.
    (fn : u -> optionm'::option' v)
    (stream : t)
    : infer' v
    =
    := fn = ~~ fn
    !\($'"futures::stream::StreamExt::filter_map(!stream, |x| async { !fn(x) })"')

:! try_join_all t =
    `(
        rust::rust_type
            `($'futures_future_TryJoinAll<`t>')
            `($'futures::future::TryJoinAll<`t>')
            "type futures_future_TryJoinAll<'T> = class end"
    )

:= try_join_all \* t.
    (x : am'::vec (future_pin (resultm::result' t sm'::std_string)))
    : try_join_all (future_pin (resultm::result' t sm'::std_string))
    =
    := x = ~~ x
    !\($'"futures::future::try_join_all(!x)"')

:! fuse_tokio t =
    `(
        rust::rust_type
            `($'tokio_prelude_stream_Fuse<`t>')
            `($'tokio::prelude::stream::Fuse<`t>')
            "type tokio_prelude_stream_Fuse<'T> = class end"
    )

:== fuse' t = fuse_tokio t

:! join_all t =
    `(
        rust::rust_type
            `($'futures_future_JoinAll<`t>')
            `($'futures::future::JoinAll<`t>')
            "type futures_future_JoinAll<'T> = class end"
    )

:= join_all \* t. (x : am'::vec (future_pin t)) : join_all (future_pin t) =
    := x = ~~ x
    !\($'"futures::future::join_all(!x)"')

:== join_handle' t = join_handle_tokio t

:= try_await_all \* t.
    (x : try_join_all (future_pin (resultm::result' t sm'::std_string)))
    : resultm::result' (am'::vec t) sm'::std_string
    =
    !\($'"!x.await"')

:= await \* t. (x : future_pin t) : t =
    !\($'"!x.await"')

:= await_send \* t. (x : future_pin_send t) : t =
    !\($'"!x.await"')

:== into_iter t = array_base t

:= into_par_iter \* t. (x : am'::vec t) : into_iter t =
    x |> am'::from_vec_base

:= par_map \* t u. (fn : t -> u) (ar : into_iter t) : into_iter u =
    := n : i32 = $'(!ar.borrow().len() as i32)'
    :~ ^ loop (i : i32) (acc : am'::vec u) : am'::vec u =
        ? i < n ?> loop (i + 1) (acc |> am'::vec_push (fn (am'::index_base i ar))) |: acc
    loop 0 (am'::new_vec []) |> am'::from_vec_base

:= par_collect \* t. (iter : into_iter t) : am'::vec t =
    iter |> am'::to_vec

:= future_init \* t. (move : bool) (x : () -> t) : infer' t =
    (!\($'"true; let __future_init = Box::pin(/*"') : bool) |> ignore
    ? move
    ?> (!\($'"*/ async move { /*"') : bool) |> ignore
    |: (!\($'"*/ async { /*"') : bool) |> ignore
    (!\($'"*/ ()"') : bool) |> ignore

    := x' = x ()

    := depth = 1, 0

    x' |> rust::fix_closure depth

    !\($'"__future_init"')

:= new_future \* t. (x : () -> t) : future_pin t =
    := result = future_init !- x
    !\($'"!result"')

:= new_future_move \* t. (x : () -> t) : future_pin t =
    := result = future_init !+ x
    !\($'"!result"')

:= new_future_send \* t. (x : () -> t) : future_pin_send t =
    := result = future_init !- x
    !\($'"!result"')

:= new_future_move_send \* t. (x : () -> t) : future_pin_send t =
    := result = future_init !+ x
    !\($'"!result"')

:! async_python t =
    `(
        backend_switch `(()) `({}) {
            Python = (\ () => global "import asyncio") : () -> ()
        }
        $'' : $'any'
    )
:== async_switch t =
    {
        Fsharp : $'Async<`t>'
        Python : async_python t
        Rust : $'std::pin::Pin<Box<dyn std::future::Future<Output = `t>>>'
    }
:! async t = $'backend_switch `(async_switch t)'

:! task t =
    `(
        ?: t ?|
        | () => $'' : $'System.Threading.Tasks.Task'
        | _ => $'' : $'System.Threading.Tasks.Task<`t>'
    )

:= new_task \* t. (fn : () -> t) : task t =
    run_target_args' fn \|
        | Fsharp _ => \ fn =>
            := result : optionm'::option' (task t) = optionm'::none' ()
            $'let mutable _new_task_!result = !result '
            $'task {'
            fn () |> ignore
            $'}'
            $'|> fun x -> _new_task_!result <- Some x'
            $'match _new_task_!result with Some x -> x | None -> failwith "async.new_task / _new_task_!result=None"'
        | _ => \ _ => null ()

:= await_task \* t. (a : task t) : async t =
    run_target \|
        | Fsharp _
        | TypeScript _ => \ () =>
            a |> $'Async.AwaitTask'
        | _ => \ () => null ()

:= future_of_async \* t. (a : async t) : future_pin t =
    $'!a' : future_pin t

:= async_of_future \* t. (f : future_pin t) : async t =
    $'!f' : async t

:! poll t =
    `(
        rust::rust_type
            `($'std_task_Poll<`t>')
            `($'std::task::Poll<`t>')
            "type std_task_Poll<'T> = class end"
    )

:! context =
    `(
        rust::rust_type
            `($'std_task_Context')
            `($'std::task::Context')
            "type std_task_Context = class end"
    )

:! waker =
    `(
        rust::rust_type
            `($'std_task_Waker')
            `($'std::task::Waker')
            "type std_task_Waker = class end"
    )

:! thread_handle =
    `(
        rust::rust_type
            `($'std_thread_Thread')
            `($'std::thread::Thread')
            "type std_thread_Thread = class end"
    )

:= current_thread () : thread_handle =
    $'std::thread::current()' : thread_handle

:= thread_park () : () =
    $'std::thread::park()' : ()

:= thread_waker_types () =
    global "struct SpiralThreadWaker(std::thread::Thread);"
    global "impl std::task::Wake for SpiralThreadWaker { fn wake(self: std::sync::Arc<Self>) { self.0.unpark() } }"

:= thread_waker (thread : thread_handle) : waker =
    thread_waker_types ()
    $'std::task::Waker::from(std::sync::Arc::new(SpiralThreadWaker(!thread)))' : waker

:= context_of_waker (w : rust::ref waker) : context =
    $'std::task::Context::from_waker(!w)' : context

:= poll_once \* t. (f : rust::ref (rust::mut' (future_pin t))) (cx : rust::ref (rust::mut' context)) : poll t =
    $'std::future::Future::poll((!f).as_mut(), !cx)' : poll t

:= poll_ready \* t. (x : t) : poll t =
    $'std::task::Poll::Ready(!x)' : poll t

:= poll_pending \* t. () : poll t =
    $'std::task::Poll::Pending' : poll t

:= poll_option \* t. (p : poll t) : optionm'::option' t =
    $'match !p { std::task::Poll::Ready(v) => Some(v), std::task::Poll::Pending => None }' : optionm'::option' t

:= poll_fn \* t. (body : rust::ref (rust::mut' context) -> poll t) : future_pin t =
    (!\($'"true; let __poll_fn = Box::pin(std::future::poll_fn(move |__poll_cx: &mut std::task::Context| { //"') : bool)
    |> ignore
    (!\\(body ($'__poll_cx' : rust::ref (rust::mut' context)), $'"true; $0 }))"') : bool) |> ignore
    !\($'"__poll_fn"')

:= catch_unwind_poll \* t.
    (f : rust::ref (rust::mut' (future_pin t)))
    (cx : rust::ref (rust::mut' context))
    : rust_caught (poll t)
    =
    $'std::panic::catch_unwind(std::panic::AssertUnwindSafe(|| std::future::Future::poll((!f).as_mut(), !cx)))'
    : rust_caught (poll t)

:= run_synchronously_rust \* t. (a : async t) : t =
    := waker = current_thread () |> thread_waker
    := context = waker |> rust::to_ref |> context_of_waker
    := future = a |> future_of_async
    future |> rust::to_mut
    context |> rust::to_mut
    := result = mut (optionm'::none' ())
    rust::loop 0 \ () =>
        := polled = poll_once (rust::to_ref_mut future) (rust::to_ref_mut context) |> poll_option
        result <- polled
        := _ =
            ? optionm'::is_none' polled ?>
                thread_park ()
                !-
            |:
                rust::break ()
                !+
        ()
    *result |> optionm'::unwrap

:= caught_poll \* t. (caught : rust_caught (poll t)) : poll (resultm::result' t exn) =
    ? rust_caught_ok caught ?>
        := polled = caught |> rust_caught_value |> poll_option
        ? optionm'::is_none' polled ?> poll_pending () |: polled |> optionm'::unwrap |> resultm::ok'' |> poll_ready
    |: caught |> rust_caught_payload |> rust_panic_text |> rust_exn_of_text |> resultm::err |> poll_ready

:= ignore \* t. (a : async t) : async () =
    ? is_rust ()
    ?> new_future_move (\ () => a |> future_of_async |> await |> ignore) |> async_of_future
    |:
        run_target \|
            | Fsharp _
            | TypeScript _ => \ () =>
                a |> $'Async.Ignore'
            | _ => \ () => null ()

:= run_synchronously \* t. (a : async t) : t =
    run_target \|
        | Fsharp _ => \ () =>
            a |> $'Async.RunSynchronously'
        | Cuda => \ () =>
            $'asyncio.run(!a())'
        | Rust => \ () => run_synchronously_rust a
        | _ => \ () => null ()

:= catch_rust \* t. (a : async t) : async (resultm::result' t exn) =
    := future = a |> future_of_async
    future |> rust::to_mut
    poll_fn \ cx => rust_quietly (\ () => catch_unwind_poll (rust::to_ref_mut future) cx |> caught_poll)
    |> async_of_future

:= start (a : async ()) : () =
    ? is_rust ()
    ?> a |> run_synchronously
    |:
        run_target \|
            | Fsharp _
            | TypeScript _ => \ () =>
                a |> $'Async.Start'
            | _ => \ () => null ()

:! task_canceled_exception = $'System.Threading.Tasks.TaskCanceledException'

:= sleep (ms : i32) : async () =
    run_target \|
        | Fsharp _
        | TypeScript _ => \ () =>
            ms |> $'Async.Sleep'
        | Cuda _ => \ () =>
            $'asyncio.sleep(!ms / 1000)'
        | Rust => \ () =>
            new_future_move (\ () => threading::sleep ms) |> async_of_future
        | _ => \ () => null ()

:= do (a : async ()) : () =
    backend_switch {
        Fsharp = \ () => $'do\! !a ' : ()
        Python = \ () => $'await !a ' : ()
    }

:= let' \* t. (a : async t) : t =
    $'let\! !a = !a '
    $'!a '

:= return_await' \* t. (a : async t) : t =
    backend_switch {
        Fsharp = \ () => $'return\! !a ' : ()
        Python = \ () => $'await !a()' : ()
    }

:= catch' \* t e. (a : async t) : async (choice2' t e) =
    ? is_rust ()
    ?>
        := r = a |> catch_rust
        $'(!r)' : async (choice2' t e)
    |:
        run_target \|
            | Fsharp _
            | TypeScript _ => \ () =>
                a |> $'Async.Catch'
            | _ => \ () => null ()

:= cancellation_token () : async threading::cancellation_token =
    $'Async.CancellationToken'

:! value_task = $'System.Threading.Tasks.ValueTask'

:= value_task_as_task (task : value_task) : task () =
    run_target \|
        | Fsharp => \ () => $'!task.AsTask' ()
        | _ => \ () => null ()

:= main () =
    ()

#### run_synchronously_drives_a_future_to_its_value (test)

In [1]:
///- --test --symbol
///> rust

:= ~n = 41i32
new_future_move (\ () => n + 1i32) |> async_of_future |> run_synchronously |> _assert_eq 42i32

__assert_eq { expected = 42 }

0

#### sleep_then_ignore_completes (test)

In [2]:
///- --test --symbol
///> rust

:= ~ms = 1i32
:= ~n = 3i32
sleep ms |> ignore |> run_synchronously
n |> _assert_eq 3i32

__assert_eq { expected = 3 }

0

In [3]:
:= async_outcome (r : resultm::result' i32 exn) : string =
    ?= r |> resultm::unbox ?|
    | Ok v => sm'::obj_to_string v
    | Error _ => "error"

#### catch_rust_turns_a_panic_into_an_error (test)

In [4]:
///- --test --symbol
///> rust

:= ~ok = 7i32
:= ~message = "boom"
:= fine = new_future_move (\ () => ok) |> async_of_future |> catch_rust |> run_synchronously
:= failed =
    new_future_move (\ () => failwith message : i32) |> async_of_future |> catch_rust |> run_synchronously
(async_outcome fine, async_outcome failed) |> _assert_eq ("7", "error")

__assert_eq { expected = (7, error) }

0